In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Annotated
from typing_extensions import TypedDict
import json

from langchain import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import AgentExecutor, Tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

import psycopg2
from configuration.config import Config

In [ ]:
import psycopg2.extras
def fetch_stock_data(symbol):
    conn = psycopg2.connect(
        host=Config.get_host_name(),
        dbname=Config.get_database_name(),
        user=Config.get_username(),
        password=Config.get_password(),
        port=Config.get_port_id(),
        sslmode='require'
    )
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)


    query = """
    SELECT date, open, high, low, close, volume 
    FROM public.daily_data
    WHERE symbol = %s
    AND date >= '2023-01-01'
    ORDER BY date ASC
    """
    cur.execute(query, (symbol,))
    rows = cur.fetchall()
    cur.close()
    conn.close()

    df = pd.DataFrame(rows, columns=['date', 'open', 'high', 'low', 'close', 'volume'])
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    return df

In [4]:
class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]
    symbol: str
    llm: ChatOpenAI
    results: Dict

In [14]:
fetch_stock_data("SBL")

Host:  {'wft-dev-postgres.postgres.database.azure.com'}
Db Name:  {'tradingviewdev'}
user:  {'tradingview'}
password:  {'Trade#ks@123'}
port:  {5432}
<cursor object at 0x0000019783DF7F20; closed: 0>


,open,high,low,close,volume
date,,,,,
2023-01-01,302.00,304.10,296.00,297.00,45032.00
2023-01-02,300.00,302.00,297.00,297.30,35179.00
2023-01-03,300.00,302.00,296.00,298.00,40533.00
2023-01-04,300.00,303.00,297.10,303.00,56117.00
2023-01-05,304.00,309.00,299.00,300.00,59839.00
...,...,...,...,...,...
2025-07-15,356.00,377.00,356.00,374.10,468205.00
2025-07-16,379.50,403.00,375.00,383.77,567833.00
2025-07-17,391.30,408.00,383.20,387.23,450633.00


In [6]:
def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    """Calculate RSI indicator"""
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

In [7]:
from decimal import Decimal

def make_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(i) for i in obj]
    elif isinstance(obj, (Decimal, np.float64, np.float32, np.int64, np.int32)):
        return float(obj)
    else:
        return obj

In [8]:
import pandas as pd
import numpy as np
import ta
from langchain import LLMChain, PromptTemplate

# Assume you have imported or defined fetch_stock_data, calculate_rsi, and make_json_serializable

def detect_fractal_zones(df, threshold=0.05, min_range_width=5.0, recent_days=90):
    df = df.copy()
    df_recent = df.tail(recent_days).reset_index()

    highs = df_recent['high'].tolist()
    lows = df_recent['low'].tolist()
    dates = df_recent['date'].tolist() if 'date' in df_recent.columns else df_recent.index.tolist()

    def get_fractals(highs, lows, dates):
        supports, resistances = [], []
        for i in range(2, len(highs)-2):
            if highs[i] > highs[i-2] and highs[i] > highs[i-1] and highs[i] > highs[i+1] and highs[i] > highs[i+2]:
                resistances.append((highs[i], dates[i]))
            if lows[i] < lows[i-2] and lows[i] < lows[i-1] and lows[i] < lows[i+1] and lows[i] < lows[i+2]:
                supports.append((lows[i], dates[i]))
        return supports, resistances

    supports_raw, resistances_raw = get_fractals(highs, lows, dates)

    def cluster_zones(levels_with_dates):
        clusters = []
        for price, date in sorted(levels_with_dates, key=lambda x: x[0]):
            added = False
            for cluster in clusters:
                avg_price = sum(p for p, _ in cluster) / len(cluster)
                if abs(price - avg_price) / price < threshold:
                    cluster.append((price, date))
                    added = True
                    break
            if not added:
                clusters.append([(price, date)])

        final_zones = []
        for cluster in clusters:
            prices = [p for p, _ in cluster]
            dates = [d for _, d in cluster]
            low, high = min(prices), max(prices)
            if len(prices) >= 2:
                recency = max(dates)
                final_zones.append({'range': (round(low, 2), round(high, 2)), 'recency': recency})
        return sorted(final_zones, key=lambda x: x['recency'], reverse=True)

    support_zones = cluster_zones(supports_raw)[:3]
    resistance_zones = cluster_zones(resistances_raw)[:3]

    return support_zones, resistance_zones

def format_zone_ranges(zones):
    return ", ".join([f"{z['range'][0]:.2f} - {z['range'][1]:.2f}" for z in zones])

In [9]:
import pandas as pd
import numpy as np
import ta
from langchain import LLMChain, PromptTemplate

def technical_analysis(state: dict) -> dict:
    symbol = state["symbol"]
    llm = state["llm"]

    hist = fetch_stock_data(symbol)

    # Clean and prepare data
    hist['high'] = pd.to_numeric(hist['high'], errors='coerce')
    hist['low'] = pd.to_numeric(hist['low'], errors='coerce')
    hist['close'] = pd.to_numeric(hist['close'], errors='coerce')
    hist['volume'] = pd.to_numeric(hist['volume'], errors='coerce')
    hist.fillna(method='ffill', inplace=True)
    hist.fillna(method='bfill', inplace=True)

    # Technical indicators
    sma_20 = hist['close'].rolling(window=20).mean()
    sma_50 = hist['close'].rolling(window=50).mean()
    rsi = calculate_rsi(hist['close'])

    exp1 = hist['close'].ewm(span=12, adjust=False).mean()
    exp2 = hist['close'].ewm(span=26, adjust=False).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=9, adjust=False).mean()

    std_20 = hist['close'].rolling(window=20).std()
    upper_band = sma_20 + (2 * std_20)
    lower_band = sma_20 - (2 * std_20)

    adx = ta.trend.ADXIndicator(hist['high'], hist['low'], hist['close'], window=14).adx()

    low_14 = hist['low'].rolling(14).min()
    high_14 = hist['high'].rolling(14).max()
    k = 100 * ((hist['close'] - low_14) / (high_14 - low_14))
    d = k.rolling(3).mean()

    closing_prices = hist['close'].tail(30).tolist()

    # Fractal support/resistance detection
    def get_fractal_levels_with_dates(highs, lows, dates):
        support_levels, resistance_levels = [], []
        for i in range(2, len(highs) - 2):
            if all(highs[i] >= highs[j] for j in [i-2, i-1, i+1, i+2]):
                resistance_levels.append((highs[i], dates[i]))
            if all(lows[i] <= lows[j] for j in [i-2, i-1, i+1, i+2]):
                support_levels.append((lows[i], dates[i]))
        return support_levels, resistance_levels

    def cluster_levels_with_dates(levels_with_dates, threshold=0.05):
        clusters = []
        levels_with_dates = sorted(levels_with_dates, key=lambda x: x[0])
        for price, date in levels_with_dates:
            added = False
            for cluster in clusters:
                avg_price = sum(p for p, _ in cluster) / len(cluster)
                if abs(price - avg_price) / price < threshold:
                    cluster.append((price, date))
                    added = True
                    break
            if not added:
                clusters.append([(price, date)])

        zone_ranges = []
        for cluster in clusters:
            prices = [p for p, _ in cluster]
            dates = [d for _, d in cluster]
            low, high = round(min(prices), 2), round(max(prices), 2)
            if len(prices) >= 2:  # key change here
                zone_ranges.append({'range': (low, high), 'recency': max(dates)})
        return sorted(zone_ranges, key=lambda x: x['recency'], reverse=True)


    # Apply fractal detection
    recent_hist = hist.tail(90).reset_index()
    date_col = 'date' if 'date' in recent_hist.columns else recent_hist.index
    supports_fd, resistances_fd = get_fractal_levels_with_dates(
        recent_hist['high'].tolist(),
        recent_hist['low'].tolist(),
        recent_hist[date_col].tolist()
    )
    fractal_support_zones = cluster_levels_with_dates(supports_fd)
    fractal_resistance_zones = cluster_levels_with_dates(resistances_fd)

    def format_zone_ranges(zones):
        return ", ".join([f"{zone['range'][0]:.2f} - {zone['range'][1]:.2f}" for zone in zones[:3]])

    raw_data = {
        'current_price': hist['close'].iloc[-1],
        'sma_20': sma_20.iloc[-1],
        'sma_50': sma_50.iloc[-1],
        'rsi': rsi.iloc[-1],
        'volume_trend': hist['volume'].iloc[-5:].mean() / hist['volume'].iloc[-20:].mean(),
        'macd': macd.iloc[-1],
        'macd_signal': signal.iloc[-1],
        'bollinger_upper': upper_band.iloc[-1],
        'bollinger_lower': lower_band.iloc[-1],
        'adx': adx.iloc[-1],
        'stochastic_k': k.iloc[-1],
        'stochastic_d': d.iloc[-1],
        'closing_prices': closing_prices,
        'fractal_support_zones': format_zone_ranges(fractal_support_zones),
        'fractal_resistance_zones': format_zone_ranges(fractal_resistance_zones),
    }

    data = make_json_serializable(raw_data)

    prompt = PromptTemplate(
        input_variables=[
            "symbol", "current_price", "sma_20", "sma_50", "rsi",
            "volume_trend", "macd", "macd_signal", "bollinger_upper",
            "bollinger_lower", "adx", "stochastic_k", "stochastic_d",
            "closing_prices", "fractal_support_zones", "fractal_resistance_zones"
        ],
        template="""
            You are a highly skilled stock technical analyst.

            Analyze the technical data for the stock symbol **{symbol}**:

            ---
            Technical Indicators:
            - Current Price: {current_price}
            - 20-day SMA: {sma_20}
            - 50-day SMA: {sma_50}
            - RSI: {rsi}
            - Volume Trend (5-day / 20-day avg): {volume_trend:.2f}
            - MACD: {macd}
            - MACD Signal Line: {macd_signal}
            - Bollinger Upper Band: {bollinger_upper}
            - Bollinger Lower Band: {bollinger_lower}
            - ADX: {adx}
            - Stochastic %K: {stochastic_k}
            - Stochastic %D: {stochastic_d}

            Recent Closing Prices (last 30 days):
            {closing_prices}
            ---
            **Fractal-Based Support/Resistance Zones:**
            - Support Zones: {fractal_support_zones}
            - Resistance Zones: {fractal_resistance_zones}

            Provide a structured analysis with the following:

            **1. Trend Direction:**
            Interpret SMA, MACD, ADX, and Bollinger Bands to describe the market trend (bullish, bearish, sideways). Justify your analysis.

            **2. Support and Resistance Levels (Fractal-Based Only):**
            - Do NOT refer to any indicator (Bollinger Bands, RSI, MACD, SMA, ADX, etc.) in your support/resistance analysis.
            - ONLY use the **fractal-based support and resistance zones** listed above.
            - Identify at least **3 key support** and **3 key resistance ranges** (price intervals) from those zones.
            - Use the exact ranges as provided (e.g., 230.50 - 233.00), do NOT pick single prices.
            - For each range, mention how many times the price zone was tested or why it is significant (e.g., reversal zone, psychological round number).
            - Distinguish between **strong** and **weak** levels.
            - Mention **psychological levels** (e.g., round numbers like 100, 200) if applicable.
            - Present the levels in descending order of importance.

            **3. Indicator Insights:**
            - RSI status (overbought/oversold/neutral).
            - Bollinger Band squeeze or breakout.
            - MACD crossovers or divergence.
            - Stochastic crossover interpretation.
            - ADX trend strength (>25 = strong trend).

            **4. Volume & Momentum:**
            Evaluate volume trend and how it supports or contradicts price movements.

            Keep your response concise, focused, and trader-friendly.
        """
    
    )

    chain = LLMChain(llm=llm, prompt=prompt)
    analysis = chain.run(
        symbol=symbol,
        current_price=data["current_price"],
        sma_20=data["sma_20"],
        sma_50=data["sma_50"],
        rsi=data["rsi"],
        volume_trend=data["volume_trend"],
        macd=data["macd"],
        macd_signal=data["macd_signal"],
        bollinger_upper=data["bollinger_upper"],
        bollinger_lower=data["bollinger_lower"],
        adx=data["adx"],
        stochastic_k=data["stochastic_k"],
        stochastic_d=data["stochastic_d"],
        closing_prices=", ".join([f"{x:.2f}" for x in data["closing_prices"]]),
        fractal_support_zones=data["fractal_support_zones"],
        fractal_resistance_zones=data["fractal_resistance_zones"],
    )

    state["results"]["technical"] = {
        "data": data,
        "analysis": analysis
    }
    return state


In [10]:
def generate_recommendation(state: State) -> State:
    """Node for final recommendation"""
    symbol = state["symbol"]
    llm = state["llm"]
    results = state["results"]

    prompt = PromptTemplate.from_template("""
            You are a financial analyst reviewing the technical analysis for stock symbol **{symbol}**.

            Technical Analysis:
            {technical}

            Based on this analysis, provide a final investment recommendation with the following structure:

            1. **Recommendation:** Choose one of [Strong Buy, Buy, Hold, Sell, Strong Sell].
            2. **Confidence Score:** A number from 0-100% representing how confident you are in this recommendation based on the technical indicators provided.
            3. **Key Reasons:** List 2-3 concise reasons supporting your recommendation.
            4. **Risk Factors:** Mention any known risks or uncertainties that could impact the outcome.
            5. **Target Price Range:** Provide a reasonable short-to-mid-term price range (e.g., over the next few weeks), based on the analysis.

            Make the output clear, structured, and suitable for a trader or investor making decisions.
            """)

    chain = LLMChain(llm=llm, prompt=prompt)
    final_recommendation = chain.run(
        symbol=symbol,
        technical=results["technical"]["analysis"],
    )

    state["results"]["recommendation"] = final_recommendation
    return state

In [11]:
def create_analysis_graph():
    graph_builder = StateGraph(State)

    graph_builder.add_node("technical", technical_analysis)
    graph_builder.add_node("recommendation", generate_recommendation)

    # Define edges
    graph_builder.add_edge(START, "technical")
    graph_builder.add_edge("technical","recommendation")
    graph_builder.add_edge("recommendation", END)

    return graph_builder.compile()

In [12]:
class StockAdvisor:
    def __init__(self):
        self.llm = llm
        self.graph = create_analysis_graph()

    def analyze_stock(self, symbol):
        """Run complete stock analysis"""
        print(f"\nAnalyzing {symbol}")

        init_state: State = {
            "symbol": symbol,
            "llm": self.llm,
            "results": {}
        }

        final_state = self.graph.invoke(init_state)
        return final_state["results"]
    
def run_analysis(symbol: str):
    """Run stock analysis and print results"""
    advisor = StockAdvisor()
    results = advisor.analyze_stock(symbol)

    print(f"\n************* Stock Analysis Report for {symbol} **************")

    print("\n************* Technical Analysis **************")
    print(results["technical"]["analysis"])

    print("\n************* Final Recommendation **************")
    print(results["recommendation"])

    return results


In [13]:
run_analysis("SBL")


Analyzing SBL
Host:  {'wft-dev-postgres.postgres.database.azure.com'}
Db Name:  {'tradingviewdev'}
user:  {'tradingview'}
password:  {'Trade#ks@123'}
port:  {5432}
<cursor object at 0x0000019783F414F0; closed: 0>


C:\Users\1\AppData\Local\Temp\ipykernel_9788\1268174482.py:17: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hist.fillna(method='ffill', inplace=True)
C:\Users\1\AppData\Local\Temp\ipykernel_9788\1268174482.py:18: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hist.fillna(method='bfill', inplace=True)
C:\Users\1\AppData\Local\Temp\ipykernel_9788\1268174482.py:175: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)
C:\Users\1\AppData\Local\Temp\ipykernel_9788\1268174482.py:176: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  analysis = chain.run(



************* Stock Analysis Report for SBL **************

************* Technical Analysis **************
### 1. Trend Direction:
- **SMA Analysis**: The current price (393.86) is significantly above the 20-day SMA (345.851) and the 50-day SMA (314.9624), indicating a strong bullish trend.
- **MACD**: The MACD (21.42) is above the MACD Signal Line (17.39), reinforcing the bullish signal.
- **ADX**: The ADX (54.31) suggests a very strong trend, as values above 25 indicate significant trend strength.
- **Bollinger Bands**: The price is close to the upper Bollinger Band (402.66), showing the stock is currently in a bullish momentum phase but approaching overbought territory.

**Conclusion**: The market trend for SBL is **bullish**, supported by positive SMA positioning, MACD signals, and a high ADX reading.

### 2. Support and Resistance Levels (Fractal-Based Only):
#### Key Support Levels:
1. **280.20 - 301.80**: 
   - **Significance**: This zone represents a recent lower range where 

{'technical': {'data': {'current_price': 393.86,
   'sma_20': 345.851,
   'sma_50': 314.9624,
   'rsi': 80.72312083729783,
   'volume_trend': 1.672976210104527,
   'macd': 21.41902128452938,
   'macd_signal': 17.388370298363597,
   'bollinger_upper': 402.6606922420037,
   'bollinger_lower': 289.0413077579963,
   'adx': 54.31063567210905,
   'stochastic_k': 82.54320987654322,
   'stochastic_d': 76.90767487620157,
   'closing_prices': [301.66,
    302.52,
    305.01,
    304.95,
    306.77,
    308.1,
    309.79,
    307.39,
    304.19,
    304.29,
    307.43,
    305.98,
    305.58,
    310.72,
    317.29,
    329.28,
    339.48,
    338.06,
    339.85,
    342.0,
    338.15,
    343.48,
    356.96,
    365.78,
    357.73,
    374.1,
    383.77,
    387.23,
    380.29,
    393.86],
   'fractal_support_zones': '280.20 - 301.80, 261.00 - 271.50',
   'fractal_resistance_zones': '294.00 - 312.50'},
  'analysis': '### 1. Trend Direction:\n- **SMA Analysis**: The current price (393.86) is sig